# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [4]:

#Part 1

class matrix:
    def __init__(self, n, m=None):
        if isinstance(n, list):
            if not n or not all(isinstance(row, list) for row in n):
                raise ValueError("Matrix must be a nonempty list of lists")

            cols = len(n[0])
            if cols == 0 or any(len(row) != cols for row in n):
                raise ValueError("All rows must have the same number of columns")

            self.data = [row.copy() for row in n]

        else:
            if not isinstance(n, int) or not isinstance(m, int):
                raise TypeError("Dimensions must be integers")

            if n <= 0 or m <= 0:
                raise ValueError("Dimensions must be positive")

            self.data = [[0 for j in range(m)] for i in range(n)]

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("Use two indices")

            i, j = key

            if isinstance(i, slice) or isinstance(j, slice):
                rows = range(self.shape()[0])[i]
                cols = range(self.shape()[1])[j]

                if isinstance(rows, int):
                    rows = [rows]
                if isinstance(cols, int):
                    cols = [cols]

                return matrix([[self.data[r][c] for c in cols]
                               for r in rows])

            return self.data[i][j]

        if isinstance(key, slice):
            return matrix(self.data[key])

        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        else:
            if len(value) != self.shape()[1]:
                raise ValueError("Incorrect row size")
            self.data[key] = list(value)

    def assign(self, other):
        if isinstance(other, matrix):
            values = other.to_list()
        else:
            values = other

        if not isinstance(values, list) or not all(
            isinstance(row, list) for row in values
        ):
            raise TypeError("Assignment requires a matrix or list of lists")

        if len(values) != self.shape()[0] or any(
            len(row) != self.shape()[1] for row in values
        ):
            raise ValueError("Matrices must have the same shape")

        self.data = [row.copy() for row in values]

    def __repr__(self):
        return str(self.data)

    #Part 2

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        n, m = self.shape()
        return matrix([[self.data[i][j] for i in range(n)]
                       for j in range(m)])

    def row(self, n):
        return matrix([self.data[n].copy()])

    def column(self, n):
        return matrix([[row[n]] for row in self.data])

    def to_list(self):
        return [row.copy() for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        return self[m_0:m_1, n_0:n_1]

    #Part 4

    def scalarmul(self, c):
        n, m = self.shape()

        return matrix([[self.data[i][j] * c for j in range(m)]
                       for i in range(n)])

    def add(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must have the same dimensions")

        n, m = self.shape()

        return matrix([[self.data[i][j] + N.data[i][j]
                        for j in range(m)] for i in range(n)])

    def sub(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must have the same dimensions")

        n, m = self.shape()

        return matrix([[self.data[i][j] - N.data[i][j]
                        for j in range(m)] for i in range(n)])

    def mat_mult(self, N):
        n, m = self.shape()
        p, q = N.shape()

        if m != p:
            raise ValueError("Incompatible dimensions for multiplication")

        return matrix([[sum(self.data[i][k] * N.data[k][j]
                            for k in range(m))
                        for j in range(q)] for i in range(n)])

    def element_mult(self, N):
        if self.shape() != N.shape():
            raise ValueError("Matrices must have the same dimensions")

        n, m = self.shape()

        return matrix([[self.data[i][j] * N.data[i][j]
                        for j in range(m)] for i in range(n)])

    def equals(self, N):
        if not isinstance(N, matrix):
            return False

        return self.shape() == N.shape() and self.data == N.data

    #Part 5

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)

        elif isinstance(other, (int, float)):
            return self.scalarmul(other)

        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        return NotImplemented

    def __add__(self, other):
        if isinstance(other, matrix):
            return self.add(other)

        return NotImplemented

    def __sub__(self, other):
        if isinstance(other, matrix):
            return self.sub(other)

        return NotImplemented

    def __eq__(self, other):
        return self.equals(other)

#Part 3

def constant(n, m, c):
    return matrix([[float(c) for j in range(m)]
                   for i in range(n)])


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    return matrix([[1.0 if i == j else 0.0
                    for j in range(n)] for i in range(n)])


#Test 1, 2, 3

M = matrix([[1, 2, 3],
            [4, 5, 6],
            [7, 8, 9]])

print("PARTS 1, 2, AND 3")
print("Original Matrix:", M)
print("Shape:", M.shape())
print("M[0][1]:", M[0][1])
print("M[0,1]:", M[0,1])
print("Transpose:", M.transpose())
print("Row 1:", M.row(1))
print("Column 2:", M.column(2))
print("To list:", M.to_list())
print("Block:", M.block(0, 2, 0, 2))
print("Slice:", M[0:2, 0:2])

print("Zeros:", zeros(2, 3))
print("Ones:", ones(2, 3))
print("Constant:", constant(2, 2, 5))
print("Identity:", eye(3))

M2 = matrix(3, 3)
M2.assign(M)
print("Assigned Matrix:", M2)

#Test 4, 5

print("\nPARTS 4 AND 5")

M = matrix([[1, 2],
            [3, 4]])

N = matrix([[5, 6],
            [7, 8]])

print("Scalar multiplication:", M.scalarmul(2))
print("Addition:", M.add(N))
print("Subtraction:", M.sub(N))
print("Matrix multiplication:", M.mat_mult(N))
print("Element-wise multiplication:", M.element_mult(N))
print("Equals:", M.equals(N))

print("2*M:", 2 * M)
print("M*2:", M * 2)
print("M+N:", M + N)
print("M-N:", M - N)
print("M*N:", M * N)
print("M==N:", M == N)

#Part 6

print("\nPART 6")

A = matrix([[1, 2],
            [3, 4]])

B = matrix([[5, 6],
            [7, 8]])

C = matrix([[2, 0],
            [1, 3]])

I = eye(2)

#Associative property: (A*B)*C = A*(B*C)

print("\nAssociative Property")
print("(A*B)*C:", (A * B) * C)
print("A*(B*C):", A * (B * C))
print("Equal:", ((A * B) * C) == (A * (B * C)))


#Distributive property: A*(B+C) = (A*B)+(A*C)

print("\nDistributive Property")
print("A*(B+C):", A * (B + C))
print("(A*B)+(A*C):", (A * B) + (A * C))
print("Equal:", A * (B + C) == (A * B) + (A * C))


#Noncommutative property: A*B != B*A

print("\nNoncommutative Property")
print("A*B:", A * B)
print("B*A:", B * A)
print("Not Equal:", (A * B) != (B * A))


#Identity property: A*I = A

print("\nIdentity Property")
print("A*I:", A * I)
print("A:", A)
print("Equal:", (A * I) == A)


PARTS 1, 2, AND 3
Original Matrix: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
Shape: (3, 3)
M[0][1]: 2
M[0,1]: 2
Transpose: [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
Row 1: [[4, 5, 6]]
Column 2: [[3], [6], [9]]
To list: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
Block: [[1, 2], [4, 5]]
Slice: [[1, 2], [4, 5]]
Zeros: [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
Ones: [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]
Constant: [[5.0, 5.0], [5.0, 5.0]]
Identity: [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
Assigned Matrix: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

PARTS 4 AND 5
Scalar multiplication: [[2, 4], [6, 8]]
Addition: [[6, 8], [10, 12]]
Subtraction: [[-4, -4], [-4, -4]]
Matrix multiplication: [[19, 22], [43, 50]]
Element-wise multiplication: [[5, 12], [21, 32]]
Equals: False
2*M: [[2, 4], [6, 8]]
M*2: [[2, 4], [6, 8]]
M+N: [[6, 8], [10, 12]]
M-N: [[-4, -4], [-4, -4]]
M*N: [[19, 22], [43, 50]]
M==N: False

PART 6

Associative Property
(A*B)*C: [[60, 66], [136, 150]]
A*(B*C): [[60, 66], [136, 150]]
Equal: True

Distributive Propert